<a href="https://colab.research.google.com/github/munnurumahesh03-coder/kaggle-predicting-loan-payback/blob/main/06_CatBoost_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Imports for the Definitive CatBoost Run

import pandas as pd
import numpy as np
import os
import gc
import warnings
import catboost as cb
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# --- Configuration ---
warnings.filterwarnings('ignore')
SEED = 42
N_SPLITS = 5
N_TRIALS = 10 # A solid number of trials for a good result.
pd.set_option('display.max_columns', None)

# --- File Paths ---
TRAIN_PATH = '/kaggle/input/02-feature-engineering-ipynb/train_featured_v2.csv'
TEST_PATH = '/kaggle/input/02-feature-engineering-ipynb/test_featured_v2.csv'

print("Cell 1: Imports for the final CatBoost run.")
print(f"CatBoost version: {cb.__version__}")
print(f"Optuna version: {optuna.__version__}")


Cell 1: Imports for the final CatBoost run.
CatBoost version: 1.2.8
Optuna version: 4.5.0


In [ ]:
# Cell 2: Load Data & Prepare for Native CatBoost

print("--- Loading Data and Preparing for Native CatBoost ---")
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

TARGET = 'loan_paid_back'
X = train_df.drop(columns=[TARGET])
y = train_df[TARGET]
test_ids = test_df['id']
X_test = test_df.drop(columns=['id'])

categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"\nConverting {len(categorical_features)} columns to 'category' dtype...")
for col in categorical_features:
    X[col] = X[col].astype('category')
    X_test[col] = X_test[col].astype('category')
print("✅ Data types converted successfully.")

del train_df, test_df
gc.collect()

print("\nCell 2: Data correctly prepared.")


--- Loading Data and Preparing for Native CatBoost ---

Converting 6 columns to 'category' dtype...
✅ Data types converted successfully.

Cell 2: Data correctly prepared.


# **Automated Hyperparameter Tuning For CatBoost**

---



In [ ]:
# Cell 3: Define the STABLE Optuna Objective Function

def objective(trial):
    # Calculate scale_pos_weight for our imbalanced dataset
    # This is the single most important parameter we were missing.
    scale_pos_weight = y.value_counts()[0] / y.value_counts()[1]

    params = {
        'objective': 'Logloss',
        'eval_metric': 'Logloss', # CRITICAL: Use Logloss for GPU stability during fit.
        'task_type': 'GPU',
        'random_seed': SEED,
        'iterations': 2000,
        'verbose': 0,
        'scale_pos_weight': scale_pos_weight, # CRITICAL: Handle imbalance.

        # Hyperparameters to be tuned by Optuna
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0, log=True),
    }

    cv_strategy = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    oof_auc_scores = []

    for fold, (train_idx, val_idx) in enumerate(cv_strategy.split(X, y)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = cb.CatBoostClassifier(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            early_stopping_rounds=50,
            cat_features=categorical_features,
            verbose=0
        )

        # Predict probabilities and calculate the REAL metric (AUC)
        preds = model.predict_proba(X_val)[:, 1]
        auc_score = roc_auc_score(y_val, preds)
        oof_auc_scores.append(auc_score)

    # Return the average AUC score to Optuna
    average_auc = np.mean(oof_auc_scores)
    return average_auc

print("Cell 3: STABLE Optuna objective function defined successfully.")


Cell 3: STABLE Optuna objective function defined successfully.


In [ ]:
# Cell 4: Run the Final Optuna Study

print("--- Starting Final CatBoost Tuning ---")

# Create the study. We want to MAXIMIZE the AUC score.
study = optuna.create_study(direction='maximize', study_name='CatBoost_Definitive_Run')

# Run the optimization. This will show a progress bar.
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print("\n--- Tuning Complete! ---")
print(f"   Number of trials: {len(study.trials)}")
print(f"   ✅ Best CV AUC Score: {study.best_value:.6f}")
print(f"   ✅ Best Parameters found:")
for key, value in study.best_params.items():
    print(f"      {key}: {value}")

# Store the best parameters
best_params_catboost = study.best_params


[I 2025-11-21 13:40:09,490] A new study created in memory with name: CatBoost_Definitive_Run


--- Starting Final CatBoost Tuning ---


  0%|          | 0/10 [00:00<?, ?it/s]

[I 2025-11-21 13:45:01,586] Trial 0 finished with value: 0.9192919826173647 and parameters: {'learning_rate': 0.05698457475132247, 'depth': 5, 'l2_leaf_reg': 9.318820716096404}. Best is trial 0 with value: 0.9192919826173647.
[I 2025-11-21 13:49:10,851] Trial 1 finished with value: 0.9184560223539716 and parameters: {'learning_rate': 0.03207889791945708, 'depth': 4, 'l2_leaf_reg': 1.017315753569982}. Best is trial 0 with value: 0.9192919826173647.
[I 2025-11-21 13:53:52,655] Trial 2 finished with value: 0.9174667337154784 and parameters: {'learning_rate': 0.06410810587109293, 'depth': 10, 'l2_leaf_reg': 2.321848402915066}. Best is trial 0 with value: 0.9192919826173647.
[I 2025-11-21 13:59:39,995] Trial 3 finished with value: 0.9179977243420122 and parameters: {'learning_rate': 0.04356023186505553, 'depth': 9, 'l2_leaf_reg': 1.8383784867686905}. Best is trial 0 with value: 0.9192919826173647.
[I 2025-11-21 14:03:48,114] Trial 4 finished with value: 0.9179809553742965 and parameters: {'

In [ ]:
# Cell 5: Train Final Model on Full Data

print("--- Training Final CatBoost Model on 100% of the Data ---")

# We already have the best_params_catboost dictionary from the completed study.
# We just need to add back the fixed parameters for the final training run.

# Add the other necessary fixed parameters
best_params_catboost['objective'] = 'Logloss'
best_params_catboost['task_type'] = 'GPU'
best_params_catboost['random_seed'] = SEED
best_params_catboost['verbose'] = 100 # Print progress every 100 rounds

# Add the critical scale_pos_weight parameter
scale_pos_weight = y.value_counts()[0] / y.value_counts()[1]
best_params_catboost['scale_pos_weight'] = scale_pos_weight
print(f"Final model parameters: {best_params_catboost}")


# Initialize the final model with all the best parameters
final_model = cb.CatBoostClassifier(**best_params_catboost)

# Fit the model on the entire training dataset
# We don't need early stopping here, as we're using all data.
# We can estimate a good number of iterations from our CV, or just train for a solid number.
# Let's train for a fixed number of rounds, e.g., 1000.
final_model.fit(
    X, y,
    cat_features=categorical_features
)

print("\n✅ Final CatBoost model trained successfully.")


--- Training Final CatBoost Model on 100% of the Data ---
Final model parameters: {'learning_rate': 0.060002475855321896, 'depth': 5, 'l2_leaf_reg': 1.6000433909687224, 'objective': 'Logloss', 'task_type': 'GPU', 'random_seed': 42, 'verbose': 100, 'scale_pos_weight': 0.25184723094496453}
0:	learn: 0.6379178	total: 34.3ms	remaining: 34.3s
100:	learn: 0.3581715	total: 3.27s	remaining: 29.1s
200:	learn: 0.3555225	total: 6.49s	remaining: 25.8s
300:	learn: 0.3535772	total: 9.7s	remaining: 22.5s
400:	learn: 0.3522735	total: 12.9s	remaining: 19.3s
500:	learn: 0.3511450	total: 16.2s	remaining: 16.1s
600:	learn: 0.3501684	total: 19.4s	remaining: 12.9s
700:	learn: 0.3492335	total: 22.7s	remaining: 9.67s
800:	learn: 0.3483316	total: 25.9s	remaining: 6.43s
900:	learn: 0.3476512	total: 29.1s	remaining: 3.2s
999:	learn: 0.3470319	total: 32.3s	remaining: 0us

✅ Final CatBoost model trained successfully.


In [ ]:
# Cell 6: Generate Final Test Predictions

print("--- Generating predictions on the test set ---")

# Use the trained final_model to predict probabilities on the test data
test_predictions = final_model.predict_proba(X_test)[:, 1]

print("✅ Test predictions generated successfully.")
print(f"First 5 predictions: {test_predictions[:5]}")


--- Generating predictions on the test set ---
✅ Test predictions generated successfully.
First 5 predictions: [0.7369062  0.9238307  0.15443997 0.80225752 0.86903149]


In [ ]:
# Cell 7: Create and Save Submission File

print("--- Creating submission file ---")

# Load the test_df again to get the 'id' column if it's not in memory
test_ids = pd.read_csv(TEST_PATH)['id']

# Create the submission DataFrame
submission_df = pd.DataFrame({
    'id': test_ids,
    TARGET: test_predictions
})

# Save the DataFrame to a csv file
submission_df.to_csv('submission_catboost.csv', index=False)

print("✅ 'submission_catboost.csv' created successfully in /kaggle/working/")
print(submission_df.head())


--- Creating submission file ---
✅ 'submission_catboost.csv' created successfully in /kaggle/working/
       id  loan_paid_back
0  593994        0.736906
1  593995        0.923831
2  593996        0.154440
3  593997        0.802258
4  593998        0.869031
